# 3 · Shapeshift: a UI that builds itself

## The problem

Forms make the *user* do the classification: they have to find the right kind of field before they can say what they
want. What if they just typed it, *"remind me to call the dentist next tuesday at 3pm"*, and the **right input control
appeared**: a date picker with a time field, a slider, a color picker, a star rating?

The tempting way is to ask an LLM to *generate* the UI (HTML or JSON). That's slow, it can produce anything, and you
have to validate every response. But a UI doesn't need prose. It needs a **decision**: which control, and which extras.

## Why Jev

Jev is a **System One** model: it returns **typed judgments with calibrated probabilities**, in about 100 ms, for a
fraction of a cent. "Which control fits?" is a classification (`Choice`), and "does it need a time field / a confirm
step?" are yes/no flags (`Noul`). Jev answers all of them **in one request**, and then ordinary code renders a control
from a **fixed, safe set**. The model can never invent a widget you didn't build.

## What we'll build

```
"remind me to call the dentist   ──► Jev: 1 Choice + 3 Noul + 1 Score ──► policy in code          ──► render a real
 next tuesday at 3pm"                  control      = datetime_picker      • flags > 0.5 → on         HTML control
                                       needs_time   = yes                  • low confidence →
                                       needs_confirm, is_recurring          fall back to text field
                                       urgency (Score 0..2)
```

## What you'll learn
- Mixing all three Jev question types (`Choice`, `Noul`, `Score`) in **one** `system_one` call.
- Using `NoulCriteria` to pin down exactly what "yes" means, and why the **wording** of a question is the model's whole world.
- The generative-UI pattern: **typed classification + deterministic rendering**, with a low-confidence fallback.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key from the `.env` file at the repo root (`TYPESAFE_API_KEY`),
- creates the Jev client `jev`. Every judgment in this notebook goes through it,
- imports the Jev question types (`Choice`, `Noul`, `NoulCriteria` and `Score`, all used here),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so numbers are easy to eyeball.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder (TYPESAFE_API_KEY)
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=24):
    """A unicode progress bar for a 0..1 value."""
    filled = round(p * width)
    return "█" * filled + "·" * (width - filled)


print("client ready")

client ready


## Step 1 · Define the questions

Five questions, three types, all answered in one call:

| Key | Type | What it returns | What the UI does with it |
|---|---|---|---|
| `control` | `Choice` | `.choice` (one of 8 controls), `.confidence`, `.probabilities` | picks the widget |
| `needs_time` | `Noul` | `.noul` = P(yes), 0..1 | adds a time field next to a date picker |
| `needs_confirm` | `Noul` | P(yes) | adds a "confirm" step for risky actions |
| `is_recurring` | `Noul` | P(yes) | adds a "repeats" marker |
| `urgency` | `Score` | `.score` from 0 to 2, plus `.confidence` | shown as information |

**Three Jev concepts in one cell:**
- **`Choice`**: pick exactly one label from `criteria`. `.probabilities` has a value for every label (they sum to 1).
- **`Noul`**: a yes/no question; `.noul` is the probability of *yes*. Adding `criteria=NoulCriteria(true=..., false=...)` spells out exactly what counts as yes and what counts as no.
- **`Score`**: `criteria` is an **ordered rubric**: level 0 is the first entry, level 2 the last. `.score` is a *continuous* position on it, so 1.4 means "between Soon and Right now".

> **Workshop note:** `needs_time` used to say *"Does the request involve a specific time of day, not just a date?"*.
> Jev answered **yes** (around 0.9) for *"where should we meet for lunch?"*. That's a fair reading, since lunch *is* a time
> of day, but it's not what we meant. We meant "does the value the user enters include a time?". The new wording says
> that, and `NoulCriteria` lists what counts as no ("a place, a date only, ..."). Lunch now scores low, while "friday
> evening" and "at 3pm" still score high. **The wording is the specification.**

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [2]:
QUESTIONS = {
    "control": Choice(
        instructions="What single input control best fits what the user is trying to specify?",
        criteria={
            "datetime_picker": "Choosing a date and/or time",
            "slider": "Setting a number within a range, like volume or brightness",
            "color_picker": "Choosing a color",
            "location_search": "Choosing a place or address",
            "star_rating": "Giving a rating or score",
            "toggle": "Turning something on or off",
            "file_upload": "Attaching or uploading a file",
            "text_field": "Free text with none of the above",
        },
    ),
    "needs_time": Noul(
        instructions="Is a time of day part of the value the user needs to enter "
                     "(for example '3pm' or 'friday evening'), not just a date or a place?",
        criteria=NoulCriteria(
            true="The value to enter includes a clock time or a part of the day",
            false="The value is a place, a date only, a setting, a file, a rating, or a color",
        ),
    ),
    "needs_confirm": Noul(instructions="Is this an action that should ask the user to confirm before running?"),
    "is_recurring": Noul(instructions="Does the request describe something repeating or recurring?"),
    "urgency": Score(
        instructions="How urgent does the request sound?",
        criteria=["Whenever", "Soon", "Right now"],     # level 0, 1, 2
    ),
}
FLAGS = ("needs_time", "needs_confirm", "is_recurring")
print("defined", len(QUESTIONS), "questions:", list(QUESTIONS))

defined 5 questions: ['control', 'needs_time', 'needs_confirm', 'is_recurring', 'urgency']


## Step 2 · Ask once and read the raw answer

`jev.system_one(state=..., questions=...)` sends the text (the **state**) and all five questions in one request. The
answers come back sorted by type: `r.choices[...]`, `r.nouls[...]`, `r.scores[...]`.

In [3]:
r = jev.system_one(state="remind me to call the dentist next tuesday at 3pm", questions=QUESTIONS)

ctrl = r.choices["control"]
print(f"control      : {ctrl.choice}  (confidence {ctrl.confidence:.2f})")
for label, p in sorted(ctrl.probabilities.items(), key=lambda kv: kv[1], reverse=True)[:3]:
    print(f"                 {label:<16} {bar(p, 16)} {p:.2f}")
for k in FLAGS:
    print(f"{k:<13}: {bar(r.nouls[k].noul, 16)} {r.nouls[k].noul:.2f}")
u = r.scores["urgency"]
print(f"urgency      : {u.score:.2f} on the 0..2 scale  (confidence {u.confidence:.2f})")
print("input tokens :", r.usage.input_tokens)

control      : datetime_picker  (confidence 1.00)
                 datetime_picker  ████████████████ 1.00
                 slider           ················ 0.00
                 color_picker     ················ 0.00
needs_time   : ████████████████ 0.98
needs_confirm: █████··········· 0.33
is_recurring : █··············· 0.05
urgency      : 0.52 on the 0..2 scale  (confidence 0.28)
input tokens : 618


**What to notice**
- `control` is a confident `datetime_picker`, and `needs_time` is near 1 because the text says "at 3pm".
- `is_recurring` is low: it's a one-off reminder. `needs_confirm` stays below the 0.5 line too: a reminder isn't risky.
- `urgency` sits low on the 0..2 scale: next Tuesday is not "right now".
- Three answer types, five answers, **one** request.

## Step 3 · The policy, in plain Python

Jev only produced numbers. Every UI decision happens here:

| Signal | Rule | Result |
|---|---|---|
| each flag | `noul > FLAG_ON` (0.5) | the flag is on |
| control confidence | `< MIN_CONF` (0.5) | **fall back to `text_field`**: a plain text box always works |
| otherwise | | use Jev's `control` |

The fallback is the safety net of generative UI: when the model isn't sure which widget fits, showing the *wrong* widget
is worse than showing a plain text box. We keep Jev's original pick in `jev_control` so you can see when the fallback fired.

The guard at the top gives a clear message if the kernel was restarted.

In [4]:
FLAG_ON = 0.5     # a yes/no flag is "on" above this
MIN_CONF = 0.5    # below this control confidence, fall back to a plain text field


def shapeshift(text, client):
    if "QUESTIONS" not in globals():
        raise RuntimeError("QUESTIONS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state=text, questions=QUESTIONS)
    ctrl = r.choices["control"]
    flags = {k: r.nouls[k].noul > FLAG_ON for k in FLAGS}
    fallback = ctrl.confidence < MIN_CONF
    return {"control": "text_field" if fallback else ctrl.choice,
            "jev_control": ctrl.choice, "confidence": ctrl.confidence, "fallback": fallback,
            "flags": flags, "time_p": r.nouls["needs_time"].noul, "urgency": r.scores["urgency"].score}


print("shapeshift() defined. FLAG_ON =", FLAG_ON, " MIN_CONF =", MIN_CONF)

shapeshift() defined. FLAG_ON = 0.5  MIN_CONF = 0.5


## Step 4 · Run it on examples

A mix of clear intents, the lunch question from the workshop note, a recurring task, a risky action, and two deliberately
**vague** requests that should trigger the fallback. For each we print the final control, Jev's confidence, the
`needs_time` probability, and the flags that are on.

In [5]:
INPUTS = [
    "remind me to call the dentist next tuesday at 3pm",
    "set the thermostat to a comfortable temperature",
    "pick an accent color for my profile",
    "where should we meet for lunch?",
    "how would you rate your support experience today?",
    "turn on dark mode",
    "attach your resume to the application",
    "water the plants every monday morning",
    "delete my account",
    "make it pop",
    "how bright and what color should the lamp be?",
]
for text in INPUTS:
    d = shapeshift(text, jev)
    on = [k for k, v in d["flags"].items() if v]
    shown = d["control"] + (f"  (fallback; Jev said {d['jev_control']})" if d["fallback"] else "")
    print(f"{text:<50} conf {d['confidence']:.2f}  time {d['time_p']:.2f}  {on}")
    print(f"{'':<50} -> {shown}")

remind me to call the dentist next tuesday at 3pm  conf 1.00  time 0.98  ['needs_time']
                                                   -> datetime_picker


set the thermostat to a comfortable temperature    conf 0.99  time 0.03  []
                                                   -> slider


pick an accent color for my profile                conf 1.00  time 0.02  []
                                                   -> color_picker


where should we meet for lunch?                    conf 1.00  time 0.14  []
                                                   -> location_search


how would you rate your support experience today?  conf 1.00  time 0.05  []
                                                   -> star_rating


...

attach your resume to the application              conf 1.00  time 0.03  []
                                                   -> file_upload


water the plants every monday morning              conf 0.74  time 0.89  ['needs_time', 'is_recurring']
                                                   -> datetime_picker


delete my account                                  conf 0.51  time 0.03  ['needs_confirm']
                                                   -> text_field


make it pop                                        conf 0.33  time 0.20  []
                                                   -> text_field  (fallback; Jev said slider)


how bright and what color should the lamp be?      conf 0.35  time 0.10  []
                                                   -> text_field  (fallback; Jev said color_picker)


**What to notice**
- Clear intents map to the obvious control with confidence near 1: thermostat → slider, accent color → color picker, rate → stars, dark mode → toggle, resume → file upload.
- **Lunch**: the control is `location_search`, and `needs_time` is now low. With the old wording it was high. (Even then it was harmless, because the time field is only drawn next to a date picker. A good reason to check *how* a flag is used before tuning it.)
- "water the plants every monday morning" turns on `is_recurring` (and `needs_time`, because "morning" is a part of the day). "delete my account" turns on `needs_confirm`.
- The two vague requests have **low confidence** (Jev splits between slider, color picker and text field), so the policy falls back to a plain text field.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired up with `ipywidgets`, and an HTML
renderer. `shapeshift()` is unchanged.

In [6]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

`control_html()` maps each control name to a **real HTML input**. This is the only place the UI's shape is decided, and
it can only ever produce one of these eight known widgets. The flags add extras: a time field (only next to a date
picker), a *confirm* badge, a *repeats* badge.

In [7]:
def control_html(d):
    c, f = d["control"], d["flags"]
    st = "font-size:15px;padding:6px 8px;border:1px solid #bbb;border-radius:6px;"
    if c == "datetime_picker":
        html = f'<input type="date" style="{st}">' + (f' <input type="time" style="{st}">' if f["needs_time"] else "")
    elif c == "slider":
        html = '<input type="range" min="0" max="100" value="50" style="width:260px;">'
    elif c == "color_picker":
        html = '<input type="color" value="#8e44c8" style="width:60px;height:40px;">'
    elif c == "location_search":
        html = f'<input type="search" placeholder="Search places..." style="{st}width:260px;">'
    elif c == "star_rating":
        html = '<span style="font-size:30px;color:#d4af37;letter-spacing:4px;">&#9734;&#9734;&#9734;&#9734;&#9734;</span>'
    elif c == "toggle":
        html = 'OFF <input type="checkbox" style="transform:scale(1.6);margin:0 8px;"> ON'
    elif c == "file_upload":
        html = f'<input type="file" style="{st}">'
    else:
        html = f'<input type="text" placeholder="type here..." style="{st}width:260px;">'
    if f["is_recurring"]:
        html += f' <span style="{BADGE}background:{AMBER};">repeats</span>'
    if f["needs_confirm"]:
        html += f' <span style="{BADGE}background:{SLATE};">+ confirm</span>'
    return html


def shapeshift_html(text):
    d = shapeshift(text, jev)
    note = ""
    if d["fallback"]:
        note = (f'<div style="color:{CORAL};font-size:12px;margin-top:6px;">low confidence {d["confidence"]:.2f} '
                f'(Jev leaned {d["jev_control"]}): fell back to a plain text field</div>')
    return box(f'<div style="{CARD}"><div style="color:{SLATE};font-size:13px;margin-bottom:10px;">'
               f'you typed: <b>{text}</b> &rarr; <code>{d["control"]}</code> ({d["confidence"]:.2f}) '
               f'&nbsp; urgency {d["urgency"]:.1f}/2</div>'
               f'<div>{control_html(d)}</div>{note}</div>')


display(HTML("".join(shapeshift_html(q) for q in [
    "remind me to call the dentist next tuesday at 3pm",
    "set the thermostat to a comfortable temperature",
    "how would you rate your support experience today?",
    "water the plants every monday morning",
    "make it pop"])))

## Step 6 · Try it yourself

Type into the box and press **Enter** or click the button (needs a live kernel). Good ones to try in a workshop:
- `book a table for friday evening` → a date picker **with** a time field ("evening" is a part of the day).
- `choose a delivery date` → a date picker **without** a time field.
- `cancel my subscription right now` → expect a *confirm* badge and a high urgency.
- `something for the weekend` → vague: watch the confidence, it's close to the fallback line.

In [8]:
text_app("describe what you want to input, e.g. 'choose a delivery date'", shapeshift_html, "Build UI")

Output()

## Recap and exercises

**Key ideas**
1. Generative UI doesn't need generated code: **typed classification + deterministic rendering** is faster, safer and testable.
2. One `system_one` call can mix `Choice`, `Noul` and `Score`; each answer comes back typed.
3. **Use confidence**: a low-confidence pick falls back to a plain text field instead of the wrong widget.
4. **The wording is the specification.** `needs_time` changed from "involves a time of day" to "the value includes a time", and `NoulCriteria` made yes/no explicit.

**Exercises**
- Raise `MIN_CONF` to 0.8 and re-run Step 4. Which inputs now fall back?
- Add a control: `"number_stepper": "Entering an exact whole number, like a quantity"`, and render it as `<input type="number">`. Try "how many tickets?".
- Put the old `needs_time` wording back and re-run Step 4. Watch the lunch line.
- Use `urgency`: give the card a red border when `urgency > 1.5`.